In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

session.sql("""
    SELECT
        CURRENT_DATABASE() AS DATABASE_NAME,
        CURRENT_SCHEMA() AS SCHEMA_NAME,
        CURRENT_WAREHOUSE() AS WAREHOUSE_NAME
""").show()

In [ ]:
%%sql -r dataframe_1
USE DATABASE FOOD_DELIVERY_ML;
USE SCHEMA ML;

In [ ]:
%%sql -r dataframe_2
SELECT
    COUNT(*) AS TOTAL_ROWS
FROM FOOD_DELIVERY_ML.ML.DELIVERY_TRAIN;

In [ ]:
%%sql -r dataframe_3
SELECT *
FROM FOOD_DELIVERY_ML.ML.DELIVERY_TRAIN
LIMIT 5;

In [ ]:
%%sql -r dataframe_5
SELECT 'DELIVERY_TRAIN' AS TABLE_NAME,
       COUNT(*) AS TOTAL_ROWS,
       COUNT(TIME_TAKEN) AS NON_NULL_TARGET
FROM FOOD_DELIVERY_ML.ML.DELIVERY_TRAIN

UNION ALL

SELECT 'DELIVERY_TRAIN_FINAL',
       COUNT(*),
       COUNT(TIME_TAKEN)
FROM FOOD_DELIVERY_ML.ML.DELIVERY_TRAIN_FINAL

UNION ALL

SELECT 'DELIVERY_TEST',
       COUNT(*),
       COUNT(TIME_TAKEN)
FROM FOOD_DELIVERY_ML.ML.DELIVERY_TEST

UNION ALL

SELECT 'DELIVERY_TEST_FINAL',
       COUNT(*),
       COUNT(TIME_TAKEN)
FROM FOOD_DELIVERY_ML.ML.DELIVERY_TEST_FINAL;

In [ ]:
%%sql -r dataframe_6
SELECT
    'CLEAN.DELIVERY_CLEAN' AS TABLE_NAME,
    COUNT(*) AS TOTAL_ROWS,
    COUNT(TIME_TAKEN) AS NON_NULL_TARGET,
    COUNT_IF(TIME_TAKEN IS NULL) AS NULL_TARGET,
    MIN(TIME_TAKEN) AS MIN_TIME,
    MAX(TIME_TAKEN) AS MAX_TIME
FROM FOOD_DELIVERY_ML.CLEAN.DELIVERY_CLEAN

UNION ALL

SELECT
    'FEATURES.DELIVERY_FEATURES',
    COUNT(*),
    COUNT(TIME_TAKEN),
    COUNT_IF(TIME_TAKEN IS NULL),
    MIN(TIME_TAKEN),
    MAX(TIME_TAKEN)
FROM FOOD_DELIVERY_ML.FEATURES.DELIVERY_FEATURES;

In [ ]:
%%sql -r dataframe_7
UPDATE FOOD_DELIVERY_ML.ML.DELIVERY_TRAIN AS T
SET TIME_TAKEN = C.TIME_TAKEN
FROM FOOD_DELIVERY_ML.CLEAN.DELIVERY_CLEAN AS C
WHERE T.ID = C.ID;

In [ ]:
%%sql -r dataframe_8
SELECT
    COUNT(*) AS TOTAL_ROWS,
    COUNT(TIME_TAKEN) AS NON_NULL_TARGET,
    COUNT_IF(TIME_TAKEN IS NULL) AS NULL_TARGET,
    MIN(TIME_TAKEN) AS MIN_TIME,
    MAX(TIME_TAKEN) AS MAX_TIME
FROM FOOD_DELIVERY_ML.ML.DELIVERY_TRAIN;

In [ ]:
%%sql -r dataframe_9
UPDATE FOOD_DELIVERY_ML.ML.DELIVERY_TEST AS T
SET TIME_TAKEN = C.TIME_TAKEN
FROM FOOD_DELIVERY_ML.CLEAN.DELIVERY_CLEAN AS C
WHERE T.ID = C.ID;

In [ ]:
%%sql -r dataframe_10
SELECT
    COUNT(*) AS TOTAL_ROWS,
    COUNT(TIME_TAKEN) AS NON_NULL_TARGET,
    COUNT_IF(TIME_TAKEN IS NULL) AS NULL_TARGET,
    MIN(TIME_TAKEN) AS MIN_TIME,
    MAX(TIME_TAKEN) AS MAX_TIME
FROM FOOD_DELIVERY_ML.ML.DELIVERY_TEST;

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

train_df = session.table("FOOD_DELIVERY_ML.ML.DELIVERY_TRAIN")

train_df.count()

In [ ]:
from snowflake.snowpark.functions import col, count, when

train_df.select(
    *[
        count(when(col(c).is_null(), 1)).alias(c)
        for c in train_df.columns
    ]
).show()

In [ ]:
from snowflake.snowpark.functions import col

train_ml_df = train_df.fillna({
    "DELIVERY_PERSON_AGE": 30,
    "ORDER_HOUR": 19,
    "PICKUP_DELAY_MIN": 10,
    "MULTIPLE_DELIVERIES": 1
})

train_ml_df.count()

In [ ]:
feature_columns = [
    "DELIVERY_PERSON_AGE",
    "DELIVERY_PERSON_RATINGS",
    "RESTAURANT_LATITUDE",
    "RESTAURANT_LONGITUDE",
    "DELIVERY_LOCATION_LATITUDE",
    "DELIVERY_LOCATION_LONGITUDE",
    "DISTANCE_KM",
    "ORDER_HOUR",
    "PICKUP_DELAY_MIN",
    "WEATHERCONDITIONS",
    "ROAD_TRAFFIC_DENSITY",
    "VEHICLE_CONDITION",
    "TYPE_OF_ORDER",
    "TYPE_OF_VEHICLE",
    "MULTIPLE_DELIVERIES",
    "FESTIVAL",
    "CITY"
]

X = train_ml_df.select(feature_columns)
y = train_ml_df.select("TIME_TAKEN")

print("Number of features:", len(feature_columns))
print("Training rows:", X.count())

In [ ]:
from snowflake.snowpark.functions import (
    count,
    min as min_,
    max as max_,
    avg,
    stddev
)

y.select(
    count("TIME_TAKEN").alias("COUNT"),
    min_("TIME_TAKEN").alias("MIN_TIME"),
    max_("TIME_TAKEN").alias("MAX_TIME"),
    avg("TIME_TAKEN").alias("MEAN_TIME"),
    stddev("TIME_TAKEN").alias("STD_TIME")
).show()

In [ ]:
test_df = session.table("FOOD_DELIVERY_ML.ML.DELIVERY_TEST")

test_ml_df = test_df.fillna({
    "DELIVERY_PERSON_AGE": 30,
    "ORDER_HOUR": 19,
    "PICKUP_DELAY_MIN": 10,
    "MULTIPLE_DELIVERIES": 1
})

X_test = test_ml_df.select(feature_columns)
y_test = test_ml_df.select("TIME_TAKEN")

print("Test rows:", X_test.count())

In [ ]:
import snowflake.ml

print("Snowflake ML is available")

In [ ]:
numeric_features = [
    "DELIVERY_PERSON_AGE",
    "DELIVERY_PERSON_RATINGS",
    "RESTAURANT_LATITUDE",
    "RESTAURANT_LONGITUDE",
    "DELIVERY_LOCATION_LATITUDE",
    "DELIVERY_LOCATION_LONGITUDE",
    "DISTANCE_KM",
    "ORDER_HOUR",
    "PICKUP_DELAY_MIN",
    "VEHICLE_CONDITION",
    "MULTIPLE_DELIVERIES"
]

categorical_features = [
    "WEATHERCONDITIONS",
    "ROAD_TRAFFIC_DENSITY",
    "TYPE_OF_ORDER",
    "TYPE_OF_VEHICLE",
    "FESTIVAL",
    "CITY"
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

In [ ]:
train_pd = train_ml_df.to_pandas()

print("Rows:", len(train_pd))
print("Columns:", len(train_pd.columns))
print(train_pd.shape)

In [ ]:
X_train_pd = train_pd[feature_columns]
y_train_pd = train_pd["TIME_TAKEN"]

print("X shape:", X_train_pd.shape)
print("y shape:", y_train_pd.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Preprocessor created successfully")

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Random Forest model created successfully")

In [ ]:
from sklearn.pipeline import Pipeline

ml_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", model)
    ]
)

print("ML pipeline created successfully")

In [ ]:
ml_pipeline.fit(X_train_pd, y_train_pd)

print("Model training completed successfully")

In [ ]:
from snowflake.ml import *

test_df = session.table("FOOD_DELIVERY_ML.ML.DELIVERY_TEST")

test_ml_df = test_df.fillna({
    "DELIVERY_PERSON_AGE": 30,
    "ORDER_HOUR": 19,
    "PICKUP_DELAY_MIN": 10,
    "MULTIPLE_DELIVERIES": 1
})

print("Test rows:", test_ml_df.count())

In [ ]:
feature_columns = [
    "DELIVERY_PERSON_AGE",
    "DELIVERY_PERSON_RATINGS",
    "RESTAURANT_LATITUDE",
    "RESTAURANT_LONGITUDE",
    "DELIVERY_LOCATION_LATITUDE",
    "DELIVERY_LOCATION_LONGITUDE",
    "DISTANCE_KM",
    "ORDER_HOUR",
    "PICKUP_DELAY_MIN",
    "WEATHERCONDITIONS",
    "ROAD_TRAFFIC_DENSITY",
    "VEHICLE_CONDITION",
    "TYPE_OF_ORDER",
    "TYPE_OF_VEHICLE",
    "MULTIPLE_DELIVERIES",
    "FESTIVAL",
    "CITY"
]

print("Features:", len(feature_columns))

In [ ]:
X_test_pd = test_pd[feature_columns]
y_test_pd = test_pd["TIME_TAKEN"]

print("Test X shape:", X_test_pd.shape)
print("Test y shape:", y_test_pd.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

numeric_features = [
    "DELIVERY_PERSON_AGE",
    "DELIVERY_PERSON_RATINGS",
    "RESTAURANT_LATITUDE",
    "RESTAURANT_LONGITUDE",
    "DELIVERY_LOCATION_LATITUDE",
    "DELIVERY_LOCATION_LONGITUDE",
    "DISTANCE_KM",
    "ORDER_HOUR",
    "PICKUP_DELAY_MIN",
    "VEHICLE_CONDITION",
    "MULTIPLE_DELIVERIES"
]

categorical_features = [
    "WEATHERCONDITIONS",
    "ROAD_TRAFFIC_DENSITY",
    "TYPE_OF_ORDER",
    "TYPE_OF_VEHICLE",
    "FESTIVAL",
    "CITY"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

ml_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", model)
    ]
)

print("Pipeline recreated successfully")

In [ ]:
ml_pipeline.fit(X_train_pd, y_train_pd)

print("Model retrained successfully")

In [ ]:
predictions = ml_pipeline.predict(X_test_pd)

print("Predictions generated:", len(predictions))
print("First 10 predictions:", predictions[:10])

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test_pd, predictions)
rmse = np.sqrt(mean_squared_error(y_test_pd, predictions))
r2 = r2_score(y_test_pd, predictions)

print("Model Evaluation")
print("----------------")
print(f"MAE  : {mae:.2f} minutes")
print(f"RMSE : {rmse:.2f} minutes")
print(f"R²   : {r2:.4f}")

In [ ]:
results_pd = X_test_pd.copy()

results_pd["ACTUAL_TIME"] = y_test_pd.values
results_pd["PREDICTED_TIME"] = predictions
results_pd["ERROR_MINUTES"] = (
    results_pd["PREDICTED_TIME"] - results_pd["ACTUAL_TIME"]
)

results_pd[[
    "ACTUAL_TIME",
    "PREDICTED_TIME",
    "ERROR_MINUTES"
]].head(10)

In [ ]:
session.write_pandas(
    results_pd,
    table_name="DELIVERY_PREDICTIONS",
    schema="ML",
    database="FOOD_DELIVERY_ML",
    auto_create_table=True,
    overwrite=True
)

print("Predictions saved to FOOD_DELIVERY_ML.ML.DELIVERY_PREDICTIONS")

In [ ]:
session.sql("""
    SELECT
        COUNT(*) AS TOTAL_PREDICTIONS,
        MIN(ACTUAL_TIME) AS MIN_ACTUAL,
        MAX(ACTUAL_TIME) AS MAX_ACTUAL,
        ROUND(AVG(ACTUAL_TIME), 2) AS AVG_ACTUAL,
        ROUND(AVG(PREDICTED_TIME), 2) AS AVG_PREDICTED
    FROM FOOD_DELIVERY_ML.ML.DELIVERY_PREDICTIONS
""").show()

In [ ]:
import joblib

joblib.dump(
    ml_pipeline,
    "food_delivery_rf_pipeline.joblib"
)

print("Model saved successfully")

In [ ]:
import os

print(os.path.exists("food_delivery_rf_pipeline.joblib"))
print(os.path.getsize("food_delivery_rf_pipeline.joblib"))

In [ ]:
put_result = session.file.put(
    "food_delivery_rf_pipeline.joblib",
    "@FOOD_DELIVERY_ML.ML.MODEL_STAGE",
    auto_compress=False,
    overwrite=True
)

print(put_result)

In [ ]:
from importlib.metadata import version

print(version("snowflake-ml-python"))

In [ ]:
import os

log_dir = "/var/log/managedservices/user/mlrs"

print("Directory exists:", os.path.exists(log_dir))
print("Directory is writable:", os.access(log_dir, os.W_OK))
print("Parent exists:", os.path.exists("/var/log/managedservices/user"))

In [ ]:
import importlib.metadata
import os

package_path = importlib.metadata.distribution("snowflake-ml-python").locate_file("snowflake/ml")

print("Snowflake ML package path:")
print(package_path)

print("\nRegistry source exists:")
registry_file = os.path.join(package_path, "registry", "registry.py")
print(os.path.exists(registry_file))
print(registry_file)

In [ ]:
import os

base = "/opt/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/site-packages/snowflake/ml"

for root, dirs, files in os.walk(base):
    for file in files:
        if file.endswith(".py"):
            path = os.path.join(root, file)
            try:
                with open(path, "r", encoding="utf-8") as f:
                    text = f.read()

                if "logs-mlrs" in text or "/var/log/managedservices" in text:
                    print("\nFOUND IN:")
                    print(path)

                    for i, line in enumerate(text.splitlines(), 1):
                        if "logs-mlrs" in line or "/var/log/managedservices" in line:
                            print(f"{i}: {line}")
            except Exception:
                pass

In [ ]:
import inspect
import snowflake.runtime.utils.logger as runtime_logger

print(inspect.getfile(runtime_logger))

In [ ]:
import inspect
import snowflake.runtime.utils.logger as runtime_logger

source = inspect.getsource(runtime_logger)

for i, line in enumerate(source.splitlines(), 1):
    if "managedservices" in line or "mlrs" in line or "CustomLoggingHandler" in line:
        print(f"{i}: {line}")

In [ ]:
import inspect
import snowflake.runtime.utils.logger as runtime_logger

source = inspect.getsource(runtime_logger.CustomLoggingHandler)

print(source)

In [ ]:
import inspect
import snowflake.runtime.utils.logger as runtime_logger

lines = inspect.getsource(runtime_logger).splitlines()

for i in range(1, 100):
    print(f"{i}: {lines[i-1]}")

In [ ]:
import tempfile
import os

tmp = tempfile.gettempdir()

print("Temp directory:", tmp)
print("Exists:", os.path.exists(tmp))
print("Writable:", os.access(tmp, os.W_OK))

In [ ]:
import snowflake.runtime.utils.logger as runtime_logger

runtime_logger.MLRS_FILE_NAME = (
    "/tmp/{log_type_name}-logs-mlrs{kernel_suffix}.log"
)

print("Temporary MLRS log path configured.")

In [ ]:
from snowflake.ml.registry import Registry

print("Model Registry import successful")

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

print("Active Snowflake session created")

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(
    session=session,
    database_name="FOOD_DELIVERY_ML",
    schema_name="ML"
)

print("Registry connection created")

In [ ]:
session.file.get(
    "@FOOD_DELIVERY_ML.ML.MODEL_STAGE/food_delivery_rf_pipeline.joblib",
    "/tmp"
)

print("Model file downloaded")

In [ ]:
import joblib

ml_pipeline = joblib.load("/tmp/food_delivery_rf_pipeline.joblib")

print(type(ml_pipeline))

In [ ]:
test_df = session.table("FOOD_DELIVERY_ML.ML.DELIVERY_TEST")

test_ml_df = test_df.fillna({
    "DELIVERY_PERSON_AGE": 30,
    "ORDER_HOUR": 19,
    "PICKUP_DELAY_MIN": 10,
    "MULTIPLE_DELIVERIES": 1
})

feature_columns = [
    "DELIVERY_PERSON_AGE",
    "DELIVERY_PERSON_RATINGS",
    "RESTAURANT_LATITUDE",
    "RESTAURANT_LONGITUDE",
    "DELIVERY_LOCATION_LATITUDE",
    "DELIVERY_LOCATION_LONGITUDE",
    "DISTANCE_KM",
    "ORDER_HOUR",
    "PICKUP_DELAY_MIN",
    "WEATHERCONDITIONS",
    "ROAD_TRAFFIC_DENSITY",
    "VEHICLE_CONDITION",
    "TYPE_OF_ORDER",
    "TYPE_OF_VEHICLE",
    "MULTIPLE_DELIVERIES",
    "FESTIVAL",
    "CITY"
]

sample_input = test_ml_df.select(feature_columns).to_pandas().head(5)

print(sample_input.shape)
print(sample_input.columns.tolist())

In [ ]:
model_ref = registry.log_model(
    ml_pipeline,
    model_name="FOOD_DELIVERY_RANDOM_FOREST",
    version_name="V1",
    comment="Random Forest model for food delivery time prediction",
    sample_input_data=sample_input
)

print("Model registered successfully")
print(model_ref)

In [ ]:
model_ref = registry.get_model(
    "FOOD_DELIVERY_RANDOM_FOREST"
).version("V1")

print(model_ref)

In [ ]:
session.sql("USE DATABASE FOOD_DELIVERY_ML").collect()
session.sql("USE SCHEMA ML").collect()

print("Database:", session.sql("SELECT CURRENT_DATABASE()").collect()[0][0])
print("Schema:", session.sql("SELECT CURRENT_SCHEMA()").collect()[0][0])

In [ ]:
model_ref_v2 = registry.log_model(
    ml_pipeline,
    model_name="FOOD_DELIVERY_RANDOM_FOREST",
    version_name="V2",
    comment="Random Forest model for food delivery time prediction - Warehouse inference",
    sample_input_data=sample_input,
    target_platforms=["WAREHOUSE"]
)

print("Warehouse-ready model registered successfully")
print(model_ref_v2)

In [ ]:
model_ref_v2 = registry.get_model(
    "FOOD_DELIVERY_RANDOM_FOREST"
).version("V2")

print(model_ref_v2)

In [ ]:
predictions_from_registry = model_ref_v2.run(
    sample_input,
    function_name="predict"
)

print(predictions_from_registry)

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE VIEW ML.DELIVERY_PREDICTION_INPUT AS
SELECT
    DELIVERY_PERSON_AGE,
    DELIVERY_PERSON_RATINGS,
    RESTAURANT_LATITUDE,
    RESTAURANT_LONGITUDE,
    DELIVERY_LOCATION_LATITUDE,
    DELIVERY_LOCATION_LONGITUDE,
    DISTANCE_KM,
    ORDER_HOUR,
    PICKUP_DELAY_MIN,
    WEATHERCONDITIONS,
    ROAD_TRAFFIC_DENSITY,
    VEHICLE_CONDITION,
    TYPE_OF_ORDER,
    TYPE_OF_VEHICLE,
    MULTIPLE_DELIVERIES,
    FESTIVAL,
    CITY
FROM ML.DELIVERY_TEST;

In [ ]:
%%sql -r dataframe_11
CREATE OR REPLACE FUNCTION FOOD_DELIVERY_ML.ML.PREDICT_DELIVERY_TIME(
    DELIVERY_PERSON_AGE FLOAT,
    DELIVERY_PERSON_RATINGS FLOAT,
    RESTAURANT_LATITUDE FLOAT,
    RESTAURANT_LONGITUDE FLOAT,
    DELIVERY_LOCATION_LATITUDE FLOAT,
    DELIVERY_LOCATION_LONGITUDE FLOAT,
    DISTANCE_KM FLOAT,
    ORDER_HOUR FLOAT,
    PICKUP_DELAY_MIN FLOAT,
    WEATHERCONDITIONS VARCHAR,
    ROAD_TRAFFIC_DENSITY VARCHAR,
    VEHICLE_CONDITION FLOAT,
    TYPE_OF_ORDER VARCHAR,
    TYPE_OF_VEHICLE VARCHAR,
    MULTIPLE_DELIVERIES FLOAT,
    FESTIVAL VARCHAR,
    CITY VARCHAR
)
RETURNS FLOAT
LANGUAGE SQL
AS
$$
    0.0::FLOAT
$$;

In [ ]:
%%sql -r dataframe_12
SELECT FOOD_DELIVERY_ML.ML.PREDICT_DELIVERY_TIME(
    30,
    4.8,
    12.9716,
    77.5946,
    12.9352,
    77.6245,
    4.2,
    19,
    10,
    'conditions Sunny',
    'High',
    2,
    'Snack',
    'motorcycle',
    1,
    'No',
    'Urban'
) AS PREDICTED_TIME;

In [ ]:
%%sql -r dataframe_13
SHOW FUNCTIONS IN MODEL FOOD_DELIVERY_ML.ML.FOOD_DELIVERY_RANDOM_FOREST VERSION V2;

In [ ]:
functions = model_ref_v2.show_functions()

for f in functions:
    print(f)

In [ ]:
%%sql -r dataframe_14
SELECT
    MODEL(
        FOOD_DELIVERY_ML.ML.FOOD_DELIVERY_RANDOM_FOREST,
        V2
    )!PREDICT(
        DELIVERY_PERSON_AGE,
        DELIVERY_PERSON_RATINGS,
        RESTAURANT_LATITUDE,
        RESTAURANT_LONGITUDE,
        DELIVERY_LOCATION_LATITUDE,
        DELIVERY_LOCATION_LONGITUDE,
        DISTANCE_KM,
        ORDER_HOUR,
        PICKUP_DELAY_MIN,
        WEATHERCONDITIONS,
        ROAD_TRAFFIC_DENSITY,
        VEHICLE_CONDITION,
        TYPE_OF_ORDER,
        TYPE_OF_VEHICLE,
        MULTIPLE_DELIVERIES,
        FESTIVAL,
        CITY
    ) AS PREDICTED_TIME
FROM FOOD_DELIVERY_ML.ML.DELIVERY_TEST
LIMIT 5;

In [ ]:
%%sql -r dataframe_18
SELECT
    COUNT_IF(DELIVERY_PERSON_AGE IS NULL) AS AGE_NULLS,
    COUNT_IF(DELIVERY_PERSON_RATINGS IS NULL) AS RATINGS_NULLS,
    COUNT_IF(RESTAURANT_LATITUDE IS NULL) AS REST_LAT_NULLS,
    COUNT_IF(RESTAURANT_LONGITUDE IS NULL) AS REST_LON_NULLS,
    COUNT_IF(DELIVERY_LOCATION_LATITUDE IS NULL) AS LOC_LAT_NULLS,
    COUNT_IF(DELIVERY_LOCATION_LONGITUDE IS NULL) AS LOC_LON_NULLS,
    COUNT_IF(DISTANCE_KM IS NULL) AS DISTANCE_NULLS,
    COUNT_IF(ORDER_HOUR IS NULL) AS ORDER_HOUR_NULLS,
    COUNT_IF(PICKUP_DELAY_MIN IS NULL) AS PICKUP_DELAY_NULLS,
    COUNT_IF(WEATHERCONDITIONS IS NULL) AS WEATHER_NULLS,
    COUNT_IF(ROAD_TRAFFIC_DENSITY IS NULL) AS TRAFFIC_NULLS,
    COUNT_IF(VEHICLE_CONDITION IS NULL) AS VEHICLE_CONDITION_NULLS,
    COUNT_IF(TYPE_OF_ORDER IS NULL) AS ORDER_TYPE_NULLS,
    COUNT_IF(TYPE_OF_VEHICLE IS NULL) AS VEHICLE_TYPE_NULLS,
    COUNT_IF(MULTIPLE_DELIVERIES IS NULL) AS MULTIPLE_DELIVERIES_NULLS,
    COUNT_IF(FESTIVAL IS NULL) AS FESTIVAL_NULLS,
    COUNT_IF(CITY IS NULL) AS CITY_NULLS
FROM FOOD_DELIVERY_ML.ML.DELIVERY_TEST;